# Clasificación de Imágenes desde Cero (Cats vs Dogs)
**Autor:** François Chollet
**Referencia:** [Keras Examples - Image classification from scratch](https://keras.io/examples/vision/image_classification_from_scratch/)

Este cuaderno demuestra cómo entrenar una red neuronal convolucional (CNN) desde cero utilizando archivos de imagen JPEG en disco, sin emplear pesos preentrenados.

## 1. Importación de dependencias

In [ ]:
import os
import numpy as np
import keras
from keras import layers
import tensorflow as tf
from tensorflow import data as tf_data
import matplotlib.pyplot as plt

print("Keras version:", keras.__version__)
print("TensorFlow version:", tf.__version__)

## 2. Descargar y preparar el dataset de Kaggle (Cats vs Dogs)
Descargamos el archivo comprimido (786 MB) y lo descomprimimos para obtener la carpeta `PetImages/`.

In [ ]:
import urllib.request
import zipfile

data_url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
zip_name = "kagglecatsanddogs_5340.zip"

if not os.path.exists("PetImages"):
    if not os.path.exists(zip_name):
        print("Descargando dataset...")
        urllib.request.urlretrieve(data_url, zip_name)
    print("Extrayendo archivo...")
    with zipfile.ZipFile(zip_name, "r") as zip_ref:
        zip_ref.extractall(".")
    print("Extracción completada.")
else:
    print("La carpeta PetImages ya existe.")

### Filtrar imágenes corruptas
En datasets de imágenes reales a menudo hay archivos corruptos. Filtramos aquellos que no incluyan la cabecera JFIF.

In [ ]:
num_skipped = 0
for folder_name in ("Cat", "Dog"):
    folder_path = os.path.join("PetImages", folder_name)
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        try:
            fobj = open(fpath, "rb")
            is_jfif = b"JFIF" in fobj.peek(10)
        finally:
            fobj.close()

        if not is_jfif:
            num_skipped += 1
            os.remove(fpath)

print(f"Se eliminaron {num_skipped} imágenes corruptas.")

## 3. Generar el `Dataset`
Utilizamos `keras.utils.image_dataset_from_directory` para crear los conjuntos de entrenamiento (80%) y validación (20%).

In [ ]:
image_size = (180, 180)
batch_size = 128

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    "PetImages",
    validation_split=0.2,
    subset="both",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size,
)

## 4. Visualización de los datos

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(images[i]).astype("uint8"))
        plt.title(int(labels[i]))
        plt.axis("off")

## 5. Aumento de datos (Data Augmentation) y Optimización del Pipeline

In [ ]:
data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
]

def data_augmentation(images):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images

# Aplicar data augmentation y prefetching en el pipeline
train_ds = train_ds.map(
    lambda img, label: (data_augmentation(img), label),
    num_parallel_calls=tf_data.AUTOTUNE,
)
train_ds = train_ds.prefetch(tf_data.AUTOTUNE)
val_ds = val_ds.prefetch(tf_data.AUTOTUNE)

## 6. Construir el modelo CNN (Estilo Xception)

In [ ]:
def make_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Bloque de entrada y normalización
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.Conv2D(128, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    previous_block_activation = x

    for size in [256, 512, 728]:
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

        residual = layers.Conv2D(size, 1, strides=2, padding="same")(
            previous_block_activation
        )
        x = layers.add([x, residual])
        previous_block_activation = x

    x = layers.SeparableConv2D(1024, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.GlobalAveragePooling2D()(x)
    units = 1 if num_classes == 2 else num_classes

    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(units, activation=None)(x)
    return keras.Model(inputs, outputs)

model = make_model(input_shape=image_size + (3,), num_classes=2)
model.summary()

## 7. Compilar y entrenar el modelo

In [ ]:
epochs = 25

callbacks = [
    keras.callbacks.ModelCheckpoint("save_at_{epoch}.keras"),
]

model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.BinaryAccuracy(name="acc")],
)

history = model.fit(
    train_ds,
    epochs=epochs,
    callbacks=callbacks,
    validation_data=val_ds,
)

model.save("final_model.keras")
print("Modelo guardado en final_model.keras")

## 8. Inferencia sobre una imagen nueva

In [ ]:
sample_image = "PetImages/Cat/6779.jpg"

if os.path.exists(sample_image):
    img = keras.utils.load_img(sample_image, target_size=image_size)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    img_array = keras.utils.img_to_array(img)
    img_array = keras.ops.expand_dims(img_array, 0)

    predictions = model.predict(img_array)
    score = float(keras.ops.sigmoid(predictions[0][0]))
    print(f"Esta imagen es {100 * (1 - score):.2f}% Gato y {100 * score:.2f}% Perro.")
else:
    print(f"No se encontró {sample_image}")